# Basic Excel Extraction Example

This is an example of how a user would perform the following steps:
- Update the user's function access to include the Excel module
- Import the istari-digital-client from PyPI
- Instantiate an instance of the istari-digital-client Client class
- Upload a Microsoft Excel file
- Extract the Excel model
- View the extracted artifacts

## Update function access

The Istari administrator must add the user to the relevant function on the Function Access page. This can be accessed in the Istari environment via Admin Panel -> Function Access -> Relevant Function -> Manage Function Access. Search to find the appropriate user, click add at the top of the window, then click save at the bottom of the window. 

## Install dependencies from PyPI

#### Note: the pip command below installs the most recent version of the digital client. Depending on your Istari release, you may need to install an older version. Refer to your relevant release page in the [release docs](https://docs.istaridigital.com/releases/2025-06-01-Release) for the appropriate SDK client version. The commented line below shows the command to install a specific client version. 

In [ ]:
!pip install istari-digital-client
# !pip install istari-digital-client==7.4.4 # use this command to install the appropriate client version if you are on an older Istari release 

!pip install python-dotenv

  Using cached deprecation-2.1.0-py2.py3-none-any.whl.metadata (4.6 kB)
  Using cached inflection-0.5.1-py2.py3-none-any.whl.metadata (1.7 kB)
  Using cached istari_digital_core-6.2.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (1.8 kB)
  Using cached pydantic-2.11.7-py3-none-any.whl.metadata (67 kB)
  Using cached python_dotenv-1.1.1-py3-none-any.whl.metadata (24 kB)
  Using cached setuptools-74.1.3-py3-none-any.whl.metadata (6.7 kB)
  Using cached urllib3-2.5.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.33.2-cp313-cp313-macosx_11_0_arm64.whl.metadata (6.8 kB)
  Using cached typing_inspection-0.4.1-py3-none-any.whl.metadata (2.6 kB)
Using cached deprecation-2.1.0-py2.py3-none-any.whl (11 kB)
Using cached inflection-0.5.1-py2.py3-none-any.whl (9.5 kB)
Using cached istari_digital_core-6.2.0-cp313-cp313-macosx_11_0_arm64.whl (2.2 MB)
Using cached pydantic-2.11.7-py3-none-any.whl (444 kB)
Using ca

In [10]:
import istari_digital_client as istari_digital
import os
from dotenv import load_dotenv
from pathlib import Path
import time
from datetime import datetime

## Instantiate the istari-digital-client Client class

To interact with Istari Digital, we need to create an instance of the istari-digital-client Client class.

The Client class takes a Configuration object that contains the following parameters:
- registry_url (required): The URL of the Istari Digital Registry Service
- registry_auth_token (required): The authentication token to use to authenticate with the Istari Digital Registry Service
- retry_enabled (optional): Whether to retry failed requests.  Defaults to True
- retry_max_attempts (optional): The maximum number of retry attempts.  Defaults to 3
- retry_min_interval_millis (optional): The minimum interval between retry attempts in milliseconds.
- retry_max_interval_millis (optional): The maximum interval between retry attempts in milliseconds.
- retry_jitter_enabled (optional): Whether to use jitter when retrying failed requests.  Defaults to True
- filesystem_cache_enabled (optional): Whether to use the filesystem cache.  Defaults to True
- filesystem_cache_root (optional): The root directory of the filesystem cache.
- filesystem_cache_clean_on_exit (optional): Whether to clean the filesystem cache on exit.  Defaults to True
- multipart_chunksize (optional): The chunk size to use when uploading files.
- multipart_threshold (optional): The threshold size to use when uploading files.

We MUST set the registry_url and registry_auth_token parameters to interact with Istari Digital.
In this example, we are following best practices and using environment variables to store the
registry_url and registry_auth_token. The environment variables are loaded using the python-dotenv package.

The configuration parameters are then used to instantiate the Configuration class.
Once instantiated, the configuration object is passed to the istari-digital-client Client class to create
an instance of the Client class.

In [3]:
load_dotenv()

dev_auth_token = os.getenv("REGISTRY_ACCESS_TOKEN")
assert dev_auth_token is not None

dev_registry_url = os.getenv("REGISTRY_URL")
assert dev_registry_url is not None

configuration = istari_digital.Configuration(
    registry_url=dev_registry_url,
    registry_auth_token=dev_auth_token,
)
assert configuration is not None

client = istari_digital.Client(
    config = configuration
)
assert client is not None

2025-07-25 13:56:43 - istari-digital-client - INFO - Logging configured with level: INFO


## Upload the Excel file to the Istari Digital Registry Service

Before extracting the Excel file, the file must be uploaded to the Istari Digital Registry Service.
To do this, we use the add_model method of the client object.
The add_model method takes the following parameters:
- path: The path to the file to upload.
- description: An optional description of the file
- version_name: An optional version name of the file
- external_identifier: An optional external identifier of the file
- display_name: An optional display name of the file.  Useful for displaying in the UI

The parameters are then passed to the add_model method of the client object to upload the file.
The add_model method returns a Model object that contains the metadata of the uploaded file.

We can then validate that the file was successfully uploaded by accessing the properties of
the Model object that is returned.

In [9]:
base_path = Path.cwd() 
file_path = base_path / "files/Excel-test-Large.xlsx"
external_identifier= "1.0.0"
display_name="Excel Test Large v1"
description="Excel Extraction Demo"
version_name = "v1"

print(f"Upload File Path: {file_path}\n")

excel_model = client.add_model(
    path=file_path,
    version_name=version_name,
    external_identifier=external_identifier,
    display_name=display_name,
    description=description
)
print(f"Uploaded base model with ID {excel_model.id}")

assert excel_model is not None
assert isinstance(excel_model, istari_digital.Model)

assert excel_model.description == description
assert excel_model.version_name == version_name
assert excel_model.external_identifier == external_identifier
assert excel_model.display_name == display_name
assert excel_model.read_bytes() == file_path.read_bytes()

Upload File Path: /Users/matthewmiller/Projects/istari-digital-client-cookbook/integrations/files/Excel-test-Large.xlsx

Uploaded base model with ID 20f47f0b-db59-42ab-bdbe-ca0c5d16c9d4


## Extract the uploaded Excel file

To extract the uploaded Excel file, the following params are needed:
- model_id: The id of the model to extract
- function: The extraction function to run
- tool_name: The tool that is needed to extract the Catia file
- tool_version: The version of the tool
- operating_system: The operating system that the tool will run on

The extract parameters are then passed to the add_job method of the client object to begin the extraction job.

Once the job is created, we poll the job until it is completed.  The job is polled by checking the JobStatusName of the job.

In [15]:
model_id = excel_model.id
function = "@istari:extract"
tool_name = "microsoft_office_excel"
tool_version = "2021"
operating_system = "Windows 11"

job = client.add_job(
    model_id=model_id,
    function=function,
    tool_name=tool_name,
    tool_version=tool_version,
    operating_system=operating_system,
)

start_time = datetime.now()
print(f"Extraction started for model ID {excel_model.id}, job ID: {job.id}")

assert job is not None
assert isinstance(job, istari_digital.Job)

while job.status.name not in [istari_digital.JobStatusName.COMPLETED, istari_digital.JobStatusName.FAILED]:
    elapsed_time = datetime.now()-start_time
    print(f"\rExtraction job {job.id} status: {job.status.name.value} ({elapsed_time})", end="")
    time.sleep(5)
    job = client.get_job(job.id)

if job.status.name.value == "Completed":
    print(f"\r\rExtraction job {job.id} completed successfully!                         ")
    model = client.get_model(job.model.id)
    print(f"\nThe following artifacts were extracted:")
    for artifact in model.artifacts:
        print(f"- artifact id: {artifact.id}")
        print(f"  revision id: {artifact.revision.id}")
        print(f"  extension: {artifact.extension}")
        print(f"  mime type: {artifact.mime}")
        print(f"  name: {artifact.name}")
else:
    print(f"\r\rExtraction job {job.id} failed with status: {job.status.name.value}")

assert job.status.name == istari_digital.JobStatusName.COMPLETED

Extraction started for model ID 20f47f0b-db59-42ab-bdbe-ca0c5d16c9d4, job ID: 7eed9c2e-3ce4-40a9-93da-0fe4d84db7a5
Extraction job 7eed9c2e-3ce4-40a9-93da-0fe4d84db7a5 completed successfully!                         

The following artifacts were extracted:
- artifact id: ba752d6d-0efd-40a0-9628-bb71d3b05387
  revision id: 2458709f-595f-4b79-a24c-659f0d4ab835
  extension: xlsx
  mime type: application/vnd.openxmlformats-officedocument.spreadsheetml.sheet
  name: workbook.xlsx
- artifact id: fbca9598-9f21-4154-a6c0-bb07bf5ab30a
  revision id: 29c1cd6e-9773-4788-ab40-db040a2e31b2
  extension: pdf
  mime type: application/pdf
  name: workbook.pdf
- artifact id: 7dbec864-fe8a-47e3-8837-454ba3604613
  revision id: 8423ae21-85bf-4584-948a-6a48c01fb5cb
  extension: png
  mime type: image/png
  name: Chart 3 UnitTest.png
- artifact id: 6767ca7a-811f-4f40-a367-38ef6bcae2aa
  revision id: cd2ad1ff-523a-40b4-bd95-eea906805560
  extension: png
  mime type: image/png
  name: Chart 4 calculate.png
- 

## Clean up the uploaded file

The uploaded file can be archived using the archive_model method of the client object.
The archive_model method takes the following parameters:
- model_id: The id of the model to archive
- archive: An Archive object that contains the reason for archiving the model

This is useful for cleaning up the files if you are going to do multiple runs of the notebook.

In [16]:
archive_reason = istari_digital.Archive(
    reason="This file was used for an example"
)

archived_model = client.archive_model(
    model_id=model_id,
    archive=archive_reason,
)

assert archived_model is not None
assert archived_model.archive_status.name == istari_digital.ArchiveStatusName.ARCHIVED